# Tutorial 4: Brain Decoding Multivariate (BDM) Analysis

## Introduction

Brain Decoding Multivariate (BDM) analysis uses machine learning to predict experimental variables (e.g., stimulus location) from EEG brain activity patterns. This reveals whether and when the brain encodes behaviorally relevant information.

### Key Concepts
- **Decoding**: Can a classifier learn to predict stimulus properties from brain activity?
- **Temporal Dynamics**: When is the stimulus information available in the EEG signal?
- **Generalization**: Does the decoding model work across different tasks or conditions?

### This Tutorial Covers
1. **Basic decoding**: Localizer task (simple, high signal)
2. **Task decoding**: Main task (more complex, realistic)
3. **Trial averaging**: Trade-off between noise reduction and sample size
4. **Cross-task decoding**: Can a model trained on localizer predict main task?

### What You'll Learn
- How to run multivariate decoding analysis
- How to interpret decoding results and timing
- How to optimize preprocessing for decoding
- When your brain data contains predictive information

## Setup: Imports and Data Loading

In [1]:
# Enable inline plotting and suppress warnings
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os

# Add open_dvm to path
sys.path.insert(0, '/Users/dvm/Documents/DvM')

# Import analysis tools
import warnings
warnings.filterwarnings('ignore')

from open_dvm.analysis import BDM
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.visualization.plot import plot_bdm_timecourse

print("✓ All imports successful!")

✓ All imports successful!


In [ ]:
# ============================================
# Configuration: Change these for your data
# ============================================
project_folder = '/Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM'
os.chdir(project_folder)

# Subject number (1-7 in this dataset)
sj = 3

# Eye-tracking quality control
eye_dict = {
    'use_tracker': True,  # Enable eye-tracking exclusion
    'window_oi': (0, 0.3),  # Window: 0-300 ms post-stimulus
    'angle_thresh': 1,  # Threshold: 1 degree visual angle
    'viewing_dist': 70,  # Viewing distance (cm)
    'screen_res': (1920, 1080),  # Screen resolution (pixels)
    'screen_h': 29,  # Screen height (cm)
    'drift_correct': (-0.2, 0)  # Drift correction window
}

# Load preprocessed data
df, epochs = FolderStructure().load_processed_epochs(
    sj, 'ses_01_main', 'main', eye_dict
)

print(f'✓ Subject {sj} data loaded')
print(f'  • {len(epochs)} trials')
print(f'  • {epochs.info["nchan"]} channels')
print(f'  • Sampling rate: {epochs.info["sfreq"]} Hz')
print(f'\nBehavioral data columns:')
print(df.columns.tolist())

In [ ]:
# Explore the data structure
print("Block types available:", df['block_type'].unique())
print("\nLocalizer block - target locations:")
print(df[df['block_type'] == 'localizer']['target_loc'].value_counts().sort_index())
print("\nMain block - target locations:")
print(df[df['block_type'] == 'main']['target_loc'].value_counts().sort_index())

---

## Part 1: Localizer Task Decoding

The localizer task is specifically designed for decoding analysis - it's a simple, repeating task with clear stimulus structure and strong neural signals.

**Question**: Can we decode the target location from EEG activity in the localizer task?

**Setup**:
- Decoder: Linear Discriminant Analysis (LDA)
- K-fold cross-validation: 10 folds
- Time window: 0-500 ms post-stimulus
- Metric: Area Under Curve (AUC)
- Exclude: Image location 8 (center, may have different neural properties)

In [ ]:
# Initialize BDM for localizer decoding
bdm_localizer = BDM(
    sj=sj,
    epochs=epochs,
    df=df,
    to_decode='target_loc',  # Decode stimulus location
    baseline=(-0.2, 0),      # Baseline correction: -200 to 0 ms
    nr_folds=10,             # 10-fold cross-validation
    elec_oi='all',           # Use all electrodes
    data_type='raw',         # Time domain (not TF)
    downsample=128           # Downsample to 128 Hz (reduce computational load)
)

print(f"✓ BDM initialized")
print(f"  • Classifier: Linear Discriminant Analysis (LDA)")
print(f"  • Decoding: target_loc (stimulus location)")
print(f"  • Electrodes: {epochs.info['nchan']} channels")
print(f"  • K-fold: 10 folds")

In [ ]:
# Run decoding on localizer task
# Note: The toolbox automatically balances class counts across blocks
output_localizer, _ = bdm_localizer.classify(
    cnds=dict(block_type=['localizer']),  # Select localizer trials
    window_oi=(-0.2, 0.5),                # Time window: -200 to 500 ms
    labels_oi='all',                      # Use all target locations
    bdm_name='localizer',
    GAT=False,                            # Standard within-time decoding
    excl_factor=dict(img_loc=[8])         # Exclude center location
)

print("✓ Localizer decoding complete")
print(f"  • Conditions: {list(output_localizer.keys())[:-1]}")
print(f"  • Time points: {output_localizer['info']['times'].shape}")

In [ ]:
# Visualize localizer decoding results
plt.figure(figsize=(12, 5))
plot_bdm_timecourse(
    output_localizer,
    colors='red',
    stats=False,
    title='Localizer Task: Target Location Decoding'
)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.3, label='Stimulus onset')
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3, label='Chance level')
plt.xlabel('Time (s)')
plt.ylabel('AUC (Area Under Curve)')
plt.legend()
plt.tight_layout()
plt.show()

# Extract key results
peak_auc = output_localizer['localizer']['dec_scores'].max()
peak_time = output_localizer['info']['times'][output_localizer['localizer']['dec_scores'].argmax()]
print(f"\n📊 Localizer Decoding Results:")
print(f"  • Peak AUC: {peak_auc:.3f} at {peak_time:.3f} s")
print(f"  • Chance level: 0.500 (8-way classification)")
print(f"  • Performance: {(peak_auc - 0.5) * 100:.1f}% above chance")

### Interpretation

The decoding results show **when** the brain encodes stimulus location information:
- **Before 0 ms**: Low AUC (pre-stimulus baseline)
- **0-150 ms**: Rising AUC as visual processing begins
- **150-400 ms**: Peak decoding performance (strong neural signal)
- **After 400 ms**: Possible decline as stimulus effects fade

**AUC Interpretation**:
- 0.50 = Chance level (no information)
- 0.60-0.70 = Weak but significant information
- 0.70-0.85 = Moderate to strong information
- 0.85+ = Very strong information

The localizer task shows strong, sustained decoding because it's optimized for consistent stimulus presentation.

---

## Part 2: Main Task Decoding

The main task is more complex and realistic. It combines:
- Target stimulus location (same as localizer)
- Distractor stimulus location (additional complexity)
- Multiple block types and attentional demands

**Question**: Does the target location decoding transfer to the more complex main task?

In [ ]:
# Initialize BDM for main task decoding
bdm_main = BDM(
    sj=sj,
    epochs=epochs,
    df=df,
    to_decode='target_loc',
    baseline=(-0.2, 0),
    nr_folds=10,
    elec_oi='all',
    data_type='raw',
    downsample=128
)

# Run decoding on main task
output_main, _ = bdm_main.classify(
    cnds=dict(block_type=['main']),  # Select main task trials
    window_oi=(-0.2, 0.5),
    labels_oi='all',
    bdm_name='main',
    GAT=False,
    excl_factor=dict(img_loc=[8])
)

print("✓ Main task decoding complete")

In [ ]:
# Compare localizer vs. main task decoding
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(output_localizer['info']['times'], output_localizer['localizer']['dec_scores'], 
         label='Localizer', color='red', linewidth=2.5)
plt.plot(output_main['info']['times'], output_main['main']['dec_scores'], 
         label='Main task', color='blue', linewidth=2.5)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
plt.xlabel('Time (s)')
plt.ylabel('AUC')
plt.title('Localizer vs. Main Task Decoding')
plt.legend()
plt.grid(True, alpha=0.3)

# Difference plot
plt.subplot(1, 2, 2)
diff = output_localizer['localizer']['dec_scores'] - output_main['main']['dec_scores']
plt.plot(output_localizer['info']['times'], diff, color='purple', linewidth=2.5)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
plt.fill_between(output_localizer['info']['times'], 0, diff, 
                  where=(diff >= 0), alpha=0.3, color='red', label='Localizer > Main')
plt.fill_between(output_localizer['info']['times'], 0, diff, 
                  where=(diff < 0), alpha=0.3, color='blue', label='Main > Localizer')
plt.xlabel('Time (s)')
plt.ylabel('AUC Difference')
plt.title('Decoding Advantage (Localizer - Main)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
peak_auc_main = output_main['main']['dec_scores'].max()
peak_time_main = output_main['info']['times'][output_main['main']['dec_scores'].argmax()]

print(f"\n📊 Main Task Decoding Results:")
print(f"  • Peak AUC: {peak_auc_main:.3f} at {peak_time_main:.3f} s")
print(f"\n📊 Comparison:")
print(f"  • Localizer peak AUC: {peak_auc:.3f}")
print(f"  • Main task peak AUC: {peak_auc_main:.3f}")
print(f"  • Difference: {peak_auc - peak_auc_main:.3f} ({(peak_auc - peak_auc_main) * 100:.1f}%)")

### Interpretation

The main task typically shows:
- **Lower peak AUC** than localizer (added complexity reduces signal clarity)
- **Similar timing** of the initial burst (core visual processing shared)
- **Different temporal profile** (complex task dynamics)

**Why is main task decoding weaker?**
1. **Dual stimuli**: Target + distractor compete for neural resources
2. **Task complexity**: Multiple cognitive demands (attention, discrimination)
3. **Variability**: More diverse behavioral patterns
4. **Sample size**: Fewer trials per condition in main task

---

## Part 3: Effect of Trial Averaging

Trial averaging reduces noise but decreases sample size. How much trial averaging is optimal?

We'll compare decoding with different trial averaging values: 1 (no averaging), 2, 4, and 6 trials.

In [ ]:
# Test effect of trial averaging on localizer decoding
plt.figure(figsize=(14, 5))
output_avg = {}

trial_avgs = [1, 2, 4, 6]
colors = ['red', 'orange', 'green', 'blue']

for tr_avg, color in zip(trial_avgs, colors):
    bdm_avg = BDM(
        sj=sj,
        epochs=epochs,
        df=df,
        to_decode='target_loc',
        baseline=(-0.2, 0),
        nr_folds=10,
        elec_oi='all',
        data_type='raw',
        downsample=128,
        avg_trials=tr_avg  # Set trial averaging
    )
    
    output, _ = bdm_avg.classify(
        cnds=dict(block_type=['localizer']),
        window_oi=(-0.2, 0.5),
        labels_oi='all',
        bdm_name=f'localizer_avg{tr_avg}',
        GAT=False,
        excl_factor=dict(img_loc=[8])
    )
    
    key = f'localizer_avg{tr_avg}'
    plt.plot(output['info']['times'], output[key]['dec_scores'], 
             label=f'{tr_avg} trial(s)', color=color, linewidth=2.5)

plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
plt.xlabel('Time (s)')
plt.ylabel('AUC')
plt.title('Effect of Trial Averaging on Decoding Performance')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Trial averaging analysis complete")

In [ ]:
# Quantify the trade-off
print("\n📊 Trial Averaging Effects:")
print("="*50)
print(f"{'Trials Averaged':<20} {'Peak AUC':<15} {'Sample Size':<15}")
print("-"*50)

n_localizer_trials = len(df[df['block_type'] == 'localizer'])

for tr_avg in trial_avgs:
    # Approximate: sample size reduces roughly by factor of trial averaging
    sample_size = n_localizer_trials // tr_avg
    # Peak AUC would come from the analysis above
    print(f"{tr_avg:<20} {'(from plot above)':<15} {sample_size:<15}")

print("-"*50)
print("\n🎯 Key Insights:")
print("  • More trial averaging → Smoother, higher AUC (less noise)")
print("  • More trial averaging → Fewer samples for training")
print("  • Optimal averaging balances noise vs. sample size")

### Trade-Off Analysis

**Benefits of trial averaging:**
- Reduces noise from single-trial variability
- Typically increases peak AUC (smoother signal)
- More stable temporal dynamics

**Costs of trial averaging:**
- Reduced sample size for cross-validation training
- May lose fine temporal resolution
- Risk of overfitting with very small sample sizes

**Recommendation**: For this dataset, trial averaging of 2-4 often provides good balance between SNR and statistical power.

---

## Part 4: Cross-Task Decoding

**Research Question**: Does the neural representation of target location learned in the **localizer** generalize to the **main task**?

We'll train a decoder on localizer trials and test it on main task trials (without retraining).

In [ ]:
# Initialize BDM with localizer as training set
bdm_cross = BDM(
    sj=sj,
    epochs=epochs.copy(),
    df=df.copy(),
    to_decode='target_loc',
    baseline=(-0.2, 0),
    nr_folds=10,
    elec_oi='all',
    data_type='raw',
    downsample=128
)

# Cross-task decoding: Train on localizer, test on main
# Format: {column: [[train_conditions], [test_conditions]]}
output_cross, _ = bdm_cross.classify(
    cnds=dict(block_type=[['localizer'], ['main']]),  # Train on localizer, test on main
    window_oi=(-0.2, 0.5),
    labels_oi='all',
    bdm_name='cross_task',
    GAT=False,
    excl_factor=dict(img_loc=[8])
)

print("✓ Cross-task decoding complete")

In [ ]:
# Compare within-task vs. cross-task decoding
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(output_localizer['info']['times'], output_localizer['localizer']['dec_scores'], 
         label='Localizer (within-task)', color='red', linewidth=2.5, linestyle='-')
plt.plot(output_main['info']['times'], output_main['main']['dec_scores'], 
         label='Main (within-task)', color='blue', linewidth=2.5, linestyle='-')
plt.plot(output_cross['info']['times'], output_cross['main']['dec_scores'], 
         label='Main (cross-task decoder)', color='green', linewidth=2.5, linestyle='--')
plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
plt.xlabel('Time (s)')
plt.ylabel('AUC')
plt.title('Within-Task vs. Cross-Task Generalization')
plt.legend()
plt.grid(True, alpha=0.3)

# Generalization gap
plt.subplot(1, 2, 2)
gap_cross = output_main['main']['dec_scores'] - output_cross['main']['dec_scores']
plt.plot(output_main['info']['times'], gap_cross, color='purple', linewidth=2.5)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
plt.fill_between(output_main['info']['times'], 0, gap_cross, 
                  where=(gap_cross >= 0), alpha=0.3, color='red', label='Within-task better')
plt.fill_between(output_main['info']['times'], 0, gap_cross, 
                  where=(gap_cross < 0), alpha=0.3, color='green', label='Cross-task better')
plt.xlabel('Time (s)')
plt.ylabel('Generalization Gap (AUC)')
plt.title('Within-Task Advantage (positive = task-specific coding)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify generalization
auc_within_main = output_main['main']['dec_scores'].max()
auc_cross = output_cross['main']['dec_scores'].max()
generalization_gap = auc_within_main - auc_cross
generalization_pct = (generalization_gap / (auc_within_main - 0.5)) * 100 if auc_within_main > 0.5 else 0

print(f"\n📊 Cross-Task Generalization Analysis:")
print(f"  • Within-task (main) peak AUC: {auc_within_main:.3f}")
print(f"  • Cross-task (localizer→main) peak AUC: {auc_cross:.3f}")
print(f"  • Generalization gap: {generalization_gap:.3f}")
print(f"  • Generalization loss: {generalization_pct:.1f}% of within-task performance")

### Interpretation

**What does cross-task generalization tell us?**

**Strong generalization (small gap)**:
- The neural representation is **task-invariant**
- Target location is encoded similarly across contexts
- Suggests robust, dedicated neural code for stimulus property

**Weak generalization (large gap)**:
- The neural representation is **task-specific**
- Main task uses different neural mechanisms (context, distractors, etc.)
- Suggests flexible, context-dependent encoding

**This is scientifically interesting!** A gap tells us the brain adapts its code based on task demands, not just encoding passive stimulus properties.

---

## Summary

You've now completed a full decoding analysis pipeline:

1. ✅ **Localizer decoding**: Simple task, strong signal, clear timing
2. ✅ **Main task decoding**: Complex task, more realistic, weaker signal
3. ✅ **Trial averaging optimization**: Balance SNR vs. sample size
4. ✅ **Cross-task generalization**: Test whether representations are task-invariant

### Key Takeaways
- **Timing matters**: Peak decoding reveals *when* information is available
- **Task matters**: Same stimulus property has different representations across contexts
- **Preprocessing matters**: Trial averaging and baseline correction significantly affect results
- **Generalization is informative**: Gaps reveal task-dependent vs. task-invariant coding

### Next Steps
- See `05_bdm_advanced.ipynb` for:
  - Generalization Across Time (GAT)
  - Time-frequency domain decoding
  - Classifier comparisons
  - Statistical testing
  - Cross-subject generalization